In [ ]:
import torch
import sqlite3
import os
import re
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from tqdm.auto import tqdm
from bitsandbytes.optim import PagedAdamW32bit
from rapidfuzz import fuzz

In [ ]:
MODEL_PATH       = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
ADAPTER_PATH     = "/mnt/storage_C1/igorzwirtes/poster_ic/lora_weights/r64q4a128_2"
SPIDER_DB_DIR    = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database"
SPIDER_TABLES    = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"
OUTPUT_DIR       = "/mnt/storage_C1/igorzwirtes/poster_ic/lora_weights/direct"
USE_BF16         = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16         = torch.cuda.is_available() and not USE_BF16
USE_CPU          = not torch.cuda.is_available()

LR               = 5e-6
NUM_EPOCHS       = 1
GRAD_ACCUM       = 4
WARMUP_STEPS     = 70
MAX_NEW_TOKENS   = 156
NUM_SAMPLES      = 12       
LOGGING_STEPS    = 1
EVAL_STEPS       = 200
SAVE_STEPS       = 200
SAVE_TOTAL_LIMIT = 3
MAX_PROMPT_TOKENS = 550    
TARGET_MAX = 350    
TARGET_MIN = 120   

In [ ]:
def get_relevant_tables(question: str, db: dict, top_k: int = 8) -> set[int]:
    q = question.lower()
    scores = {}

    for i, table in enumerate(db["table_names_original"]):
        table_score = fuzz.partial_ratio(table.lower(), q)

        cols = [c[1].lower() for c in db["column_names_original"] if c[0] == i]
        col_score = max((fuzz.partial_ratio(c, q) for c in cols), default=0)

        scores[i] = max(table_score, col_score * 0.8)

    # ranking + filtro leve
    top = sorted(scores, key=scores.get, reverse=True)
    top = [i for i in top if scores[i] > 5][:top_k]

    seed = set(top)

    # FK expansion (1-hop)
    fk_tables = set(seed)

    for fk in db.get("foreign_keys", []):
        t1 = db["column_names_original"][fk[0]][0]
        t2 = db["column_names_original"][fk[1]][0]

        if t1 in seed or t2 in seed:
            fk_tables.add(t1)
            fk_tables.add(t2)

    return fk_tables

In [ ]:
with open(SPIDER_TABLES) as f:
    tables_data = json.load(f)

def format_schema_compact(db: dict, question: str, max_cols: int = 8) -> str:
    col_names = db["column_names_original"]
    col_types = db["column_types"]
    pks       = set(db.get("primary_keys", []))

    fk_map = {}
    for fk in db.get("foreign_keys", []):
        src, ref      = fk[0], fk[1]
        ref_table     = db["table_names_original"][col_names[ref][0]]
        ref_col       = col_names[ref][1]
        fk_map.setdefault(src, []).append(f"{ref_table}.{ref_col}")

    relevant_tables = get_relevant_tables(question, db)
    q_tokens        = set(re.sub(r"[^\w\s]", "", question.lower()).split())

    lines = []
    for i, table in enumerate(db["table_names_original"]):
        if i not in relevant_tables:
            continue

        cols_data = [
            (idx, col[1], col_types[idx])
            for idx, col in enumerate(col_names)
            if col[0] == i
        ]

        def col_score(item):
            idx, name, _ = item
            score = fuzz.partial_ratio(name.lower(), question.lower())
            if any(t in name.lower() for t in q_tokens): score += 20
            if idx in pks:    score += 30
            if idx in fk_map: score += 25
            return score

        cols_sorted = sorted(cols_data, key=col_score, reverse=True)[:max_cols]
        cols_sorted.sort(key=lambda x: x[0])

        col_parts = []
        for idx, name, ctype in cols_sorted:
            part = f"{name} {ctype}"
            if idx in pks:    part += " PK"
            if idx in fk_map: part += f" FK→{fk_map[idx]}"
            col_parts.append(part)

        lines.append(f"{table}({', '.join(col_parts)})")

    return "\n".join(lines)

tables_index = {db["db_id"]: db for db in tables_data}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
dataset    = load_dataset("spider")
train_data = dataset["train"]
val_data   = dataset["validation"]

def build_prompt(example: dict) -> str:
    db = tables_index[example["db_id"]]
    q  = example["question"]

    for max_cols in [8, 6, 4, 3]:
        schema = format_schema_compact(db, q, max_cols=max_cols)

        messages = [
            {"role": "system", "content": "Convert the question to a valid SQLite query. Output SQL only. Always use table aliases when joining multiple tables."},
            {"role": "user",   "content": f"Schema ({example['db_id']}):\n{schema}\n\nQuestion: {q}"},
        ]

        prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        n_tokens = len(tokenizer.encode(prompt))

        if n_tokens <= TARGET_MAX:
            break

    # hard cap: trunca linhas do schema até caber
    if n_tokens > MAX_PROMPT_TOKENS:
        schema_lines = schema.split("\n")
        while len(schema_lines) > 1 and n_tokens > MAX_PROMPT_TOKENS:
            schema_lines.pop()
            messages[1]["content"] = f"Schema ({example['db_id']}):\n{chr(10).join(schema_lines)}\n\nQuestion: {q}"
            prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            n_tokens = len(tokenizer.encode(prompt))

    return prompt

# Testar uso máximo de VRAM
#train_data = sorted(train_data, key=lambda ex: len(tokenizer(build_prompt(ex))["input_ids"]), reverse=True)

In [ ]:
def extract_sql(text):
    # bloco markdown
    match = re.search(
        r"```(?:sql)?\s*(.*?)```",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if match:
        sql = match.group(1).strip()
    else:
        # pega primeira query SQL
        match = re.search(
            r"(SELECT|INSERT|UPDATE|DELETE|WITH)\b.*?;",
            text,
            re.DOTALL | re.IGNORECASE
        )

        if match:
            sql = match.group(0).strip()
        else:
            sql = text.strip()

    # remove comentários
    sql = re.sub(r"--.*", "", sql)

    # remove markdown sobrando
    sql = sql.replace("```", "").strip()

    return sql

DB_CONNECTIONS = {}
for db in tables_data:
    db_id = db["db_id"]
    db_path = os.path.join(SPIDER_DB_DIR, db_id, f"{db_id}.sqlite")

    conn = sqlite3.connect(db_path, check_same_thread=False)
    DB_CONNECTIONS[db_id] = conn

def execution_reward(db_id, pred_sql, gold_sql):
    conn = DB_CONNECTIONS[db_id]
    cur = conn.cursor()

    # sanity check
    try:
        cur.execute(gold_sql)
        gold_res = sorted(cur.fetchall())
    except Exception:
        return 0.0  

    try:
        cur.execute(pred_sql)
        pred_res = sorted(cur.fetchall())
    except sqlite3.OperationalError as e:
        msg = str(e).lower()
        if "syntax error" in msg:
            return -0.4   # SQL malformado — erro grave
        else:
            return -0.1   # tabela/coluna errada — estrutura ok, schema errou
    except Exception:
        return -0.5       # erro inesperado (timeout, etc.)

    if pred_res == gold_res:
        return 1.0        # exato

    # Resultado parcialmente correto
    if not gold_res or not pred_res:
        return -0.05      # um vazio e o outro não

    gold_set = set(gold_res)
    pred_set = set(pred_res)

    precision = len(pred_set & gold_set) / len(pred_set) if pred_set else 0.0
    recall    = len(pred_set & gold_set) / len(gold_set) if gold_set else 0.0

    if precision + recall == 0:
        return 0.0

    f1 = 2 * precision * recall / (precision + recall)
    return round(0.5 * f1, 4)   # parcial entre 0 e 0.5

In [ ]:
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False

base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
'''
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"      
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
'''
#model = get_peft_model(base_model, lora_config)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=True,
)

model.print_trainable_parameters()

optimizer = PagedAdamW32bit(model.parameters(), lr=LR)

total_steps = (len(train_data) * NUM_EPOCHS) // GRAD_ACCUM
scheduler   = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)

In [ ]:
def evaluate(model, val_data, max_examples=200):
    model.eval()
    correct = 0
    total   = min(max_examples, len(val_data))
    device  = next(model.parameters()).device
    for example in list(val_data)[:total]:
        prompt = build_prompt(example)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS).to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_tokens = output[0][inputs["input_ids"].shape[1]:]
        pred_sql   = extract_sql(tokenizer.decode(gen_tokens, skip_special_tokens=True))
        correct   += execution_reward(example["db_id"], pred_sql, example["query"])
    model.train()
    return correct / total

In [ ]:
device         = next(model.parameters()).device
global_step    = 0
best_eval      = 0.0
saved_checkpoints = []

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad()
    total_loss   = 0.0
    total_reward = 0.0
    window_loss   = 0.0
    window_reward = 0.0
    window_count  = 0
    step_loss_accum   = 0.0
    step_reward_accum = 0.0

    for i, example in enumerate(tqdm(train_data, desc=f"Epoch {epoch+1}", disable=False)):
        prompt = build_prompt(example)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS).to(device)

        rewards        = []
        log_probs_list = []
        entropies = []

        for _ in range(NUM_SAMPLES):
            model.eval()
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.95,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            model.train()

            gen_tokens = output[0][inputs["input_ids"].shape[1]:]
            pred_sql   = extract_sql(tokenizer.decode(gen_tokens, skip_special_tokens=True))
            pred_sql = pred_sql.split(";")[0] + ";"
            reward     = execution_reward(example["db_id"], pred_sql, example["query"])
            rewards.append(reward)
            
            logits = model(input_ids=output).logits
            shift_logits = logits[0, inputs["input_ids"].shape[1]-1:-1]
            shift_labels = gen_tokens

            log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)

            token_log_probs = log_probs[
                torch.arange(len(shift_labels), device=device),
                shift_labels
            ]

            seq_log_prob = token_log_probs.sum()

            log_probs_list.append(seq_log_prob)

            entropy = -(log_probs.exp() * log_probs).sum(dim=-1).mean()
            entropies.append(entropy)

        rewards_tensor = torch.tensor(rewards, dtype=torch.float32)

        advantages = rewards_tensor - rewards_tensor.mean()
        advantages = advantages / (advantages.std() + 1e-8)
        advantages = advantages.detach()

        mean_entropy = torch.stack(entropies).mean()

        policy_loss = torch.stack([
            -adv.to(device) * lp
            for adv, lp in zip(advantages, log_probs_list)
        ]).sum() / NUM_SAMPLES

        total_loss_step = policy_loss - 0.01 * mean_entropy

        (total_loss_step / GRAD_ACCUM).backward()

        step_loss_accum   += total_loss_step.detach().item()
        step_reward_accum += rewards_tensor.mean().detach().item()
        total_reward      += rewards_tensor.mean().detach().item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            window_loss   += step_loss_accum / GRAD_ACCUM
            window_reward += step_reward_accum / GRAD_ACCUM
            window_count  += 1

            # Reseta acumuladores do batch
            step_loss_accum   = 0.0
            step_reward_accum = 0.0

            if global_step % LOGGING_STEPS == 0:
                print(f"Step {global_step} | loss: {window_loss/window_count:.4f} | reward: {window_reward/window_count:.4f} | lr: {scheduler.get_last_lr()[0]:.2e}")
                window_loss   = 0.0
                window_reward = 0.0
                window_count  = 0

            if global_step % EVAL_STEPS == 0:
                ex = evaluate(model, val_data)
                avg_reward = total_reward / (i + 1)  
                print(f"Step {global_step} | Train reward: {avg_reward:.4f} | Val EX: {ex:.4f}")

            if global_step % SAVE_STEPS == 0:
                save_path = f"{OUTPUT_DIR}/checkpointB-{global_step}"
                model.save_pretrained(save_path)
                saved_checkpoints.append(save_path)
                if len(saved_checkpoints) > SAVE_TOTAL_LIMIT:
                    import shutil
                    shutil.rmtree(saved_checkpoints.pop(0))

    if len(train_data) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        global_step += 1

model.save_pretrained(f"{OUTPUT_DIR}/finalB")
print("Treino concluído.")